In [ ]:
%matplotlib inline

import jax
import posterior_plots
from jax import numpy as jnp
from matplotlib import pyplot as plt

import xxm.plot.core as plot
import xxm.plot.models as model_plot
from xxm import DatasetCollection
from xxm.core.align import align_procrustes as align_latent
from xxm.lds import PoissonLDS

In [ ]:
def build_lds_example(
    latent_dim=2,
    num_neurons=12,
    angle=0.15,
    radius=0.995,
    base_rate=1.0,
) -> PoissonLDS:
    """Build an LDS example with a damped rotation dynamics."""

    dynamics_matrix = radius * jnp.array(
        [
            [jnp.cos(angle), -jnp.sin(angle)],
            [jnp.sin(angle), jnp.cos(angle)],
        ]
    )

    preferred_angles = jnp.linspace(
        0.0,
        2.0 * jnp.pi,
        num_neurons,
        endpoint=False,
    )

    readout = 0.7 * jnp.stack(
        [
            jnp.cos(preferred_angles),
            jnp.sin(preferred_angles),
        ],
        axis=1,
    )

    return PoissonLDS.from_params(
        initial_mean=jnp.array([1.5, 0.0]),
        initial_covariance=0.05 * jnp.eye(latent_dim),
        dynamics_coefficients=dynamics_matrix,
        dynamics_bias=jnp.zeros(latent_dim),
        dynamics_covariance=0.015 * jnp.eye(latent_dim),
        emissions=(
            {
                'coefficients': readout,
                'bias': jnp.full((num_neurons,), jnp.log(base_rate)),
            },
        ),
    )


true_model = build_lds_example()

truth = true_model.sample(key=jax.random.key(0), steps=500)
data = DatasetCollection.from_array(truth.observations.get())


def plot_data(latents, observations):

    _, axs = plt.subplots(
        ncols=2,
        figsize=(8, 3),
        constrained_layout=True,
        gridspec_kw={'width_ratios': [1, 2]},
    )

    ax = axs[0]

    plot.plot_traces_2d(ax, latents)

    ax = axs[1]
    ax.figure.colorbar(plot.plot_traces_image(ax, observations), ax=ax, label='')


plot_data(
    truth.latents.get(),
    truth.observations.get(),
)

In [ ]:
inferred = true_model.infer(data)

In [ ]:
latent_posterior = inferred.latent_posterior().get()

posterior_plots.plot_posterior_over_time(latent_posterior, truth.latents.get())
posterior_plots.plot_occupancy(latent_posterior, truth.latents.get())

In [ ]:
initial_model = PoissonLDS.via_pca(jax.random.key(1), data, latent_dim=2).get()

In [ ]:
fitted, objective_trace = initial_model.fit(
    data,
    num_iters=100,
)

model_plot.plot_fit_progress(objective_trace)

In [ ]:
def align_inferred(inferred, true_latents):

    inferred_latents = inferred.latent_posterior().mean

    alignment = align_latent(
        inferred_latents.get(),
        true_latents.get(),
    )

    return inferred.align(alignment)


initial = initial_model.infer(data)
initial = align_inferred(initial, truth.latents)
fitted = align_inferred(fitted, truth.latents)

In [ ]:
def plot_latent_comparison(true_latents, initial, fitted):
    _, axs = plt.subplots(ncols=2, figsize=(6, 3), constrained_layout=True)

    for ax, inferred, title in zip(
        axs,
        (initial, fitted),
        ('Initial posterior', 'Fitted posterior'),
    ):
        posterior_plots.plot_occupancy(
            inferred.latent_posterior().get(),
            true_latents,
            ax=ax,
        )
        ax.set(title=title)


plot_latent_comparison(truth.latents.get(), initial, fitted)

In [ ]:
def plot_dynamics(true_model, initial_model, fitted_model):
    _, axs = plt.subplots(
        ncols=2,
        sharex='all',
        sharey='all',
        figsize=(8, 4),
    )

    model_plot.plot_dyn_linear_gaussian_comparison(
        axs[0], true_model.dynamics(), initial_model.dynamics(), color1='xkcd:grey'
    )
    model_plot.plot_dyn_linear_gaussian_comparison(
        axs[1],
        true_model.dynamics(),
        fitted_model.dynamics(),
        color1='xkcd:apple green',
    )


plot_dynamics(
    true_model,
    initial.model,
    fitted.model,
)

In [ ]:
def plot_observations_comparison(
    observations, initial_prediction, fitted_prediction, example_neuron
):
    _, axs = plt.subplots(
        nrows=4,
        sharex='all',
        constrained_layout=True,
        figsize=(8, 6),
        gridspec_kw={'height_ratios': [1, 1, 1, 2]},
    )

    ax = axs[0]

    ax.figure.colorbar(plot.plot_traces_image(ax, observations), ax=ax, label='')

    ax = axs[1]

    ax.figure.colorbar(plot.plot_traces_image(ax, initial_prediction), ax=ax, label='')

    ax = axs[2]

    ax.figure.colorbar(plot.plot_traces_image(ax, fitted_prediction), ax=ax, label='')

    ax = axs[3]
    ax.plot(
        observations[:, example_neuron],
        alpha=0.4,
        label=f'example neuron {example_neuron}',
        color='k',
    )
    ax.plot(
        initial_prediction[:, example_neuron], label='initial', color='xkcd:apple green'
    )
    ax.plot(fitted_prediction[:, example_neuron], label='fitted', color='xkcd:coral')
    ax.legend()


plot_observations_comparison(
    truth.observations.get(),
    initial_prediction=initial.observation_mean().get(),
    fitted_prediction=fitted.observation_mean().get(),
    example_neuron=1,
)